# 03. 심화: 커스터마이징 평가 루프와 안전 게이트

목표: 발표문의 self-fine-tuning 데모처럼 objective, scorer, evaluation loop를 구성하고, 모델 커스터마이징 후에도 safety와 calibration을 확인해야 함을 배웁니다.

실행 방법: 모든 셀을 순서대로 실행합니다. 외부 패키지는 필요하지 않습니다.

## 1. Lipogram objective 만들기

발표문은 알파벳 `e`를 쓰지 않는 lipogram 모델로 fine-tune하는 데모를 보여 줍니다. 여기서는 실제 모델 학습 대신 scoring function과 후보 응답 평가를 구현합니다.

In [ ]:
def lipogram_score(answer, forbidden="e"):
    # 금지 문자가 하나라도 있으면 실패로 둡니다. 실제 eval은 더 많은 품질 기준이 필요합니다.
    return 0.0 if forbidden.lower() in answer.lower() else 10.0


candidate_answers = [
    "As your team ships the model, celebrate and monitor issues.",
    "As your group puts out a big AI, thank staff, watch for bugs, and plan upcoming work.",
    "Party, thank staff, audit logs, fix faults fast, and plan what follows.",
]

for answer in candidate_answers:
    print(f"score={lipogram_score(answer):4.1f} | {answer}")

## 2. Base와 customized 후보 비교

Fine-tuning이 좋아졌는지 판단하려면 base model과 customized model을 같은 prompt set에서 비교해야 합니다. 아래는 간단한 pass rate 평가입니다.

In [ ]:
prompts = [
    "What should I do after releasing a model?",
    "Give me a short launch checklist.",
    "How should I answer user feedback?",
]

base_outputs = [
    "Celebrate, thank everyone, monitor issues, and write release notes.",
    "Review metrics, announce the release, and prepare a response plan.",
    "Listen carefully, acknowledge the concern, and explain the next step.",
]

custom_outputs = [
    "Thank staff, watch logs, fix faults fast, and plan upcoming work.",
    "Audit stats, post a launch nota, assign on-call, and track bugs.",
    "Ask for facts, sum up pain, and say what you will do now.",
]


def evaluate_outputs(outputs):
    scores = [lipogram_score(output) for output in outputs]
    return {
        "mean_score": sum(scores) / len(scores),
        "pass_rate": sum(score == 10.0 for score in scores) / len(scores),
    }


for name, outputs in [("base", base_outputs), ("custom", custom_outputs)]:
    result = evaluate_outputs(outputs)
    print(f"{name:6s} mean_score={result['mean_score']:.1f} pass_rate={result['pass_rate']:.0%}")

## 3. Calibration 확인: Brier score

모델 카드는 calibration을 중요한 epistemic 속성으로 봅니다. Brier score는 확률 예측이 실제 결과와 얼마나 맞는지 측정하는 간단한 지표입니다. 낮을수록 좋습니다.

In [ ]:
def brier_score(probabilities, outcomes):
    return sum((p - y) ** 2 for p, y in zip(probabilities, outcomes)) / len(outcomes)


calibrated_probs = [0.10, 0.30, 0.55, 0.75, 0.20]
overconfident_probs = [0.01, 0.05, 0.95, 0.98, 0.99]
outcomes = [0, 0, 1, 1, 0]

print("calibrated brier:", round(brier_score(calibrated_probs, outcomes), 3))
print("overconfident brier:", round(brier_score(overconfident_probs, outcomes), 3))

## 4. Deployment safety gate

Open weights 모델은 fine-tuning 뒤에도 application layer safeguard가 필요합니다. 아래는 high-stakes 또는 harmful request를 배포 전에 걸러내는 매우 단순한 예시입니다. 실제 제품에서는 전문 moderation model과 로그 기반 모니터링이 필요합니다.

In [ ]:
blocked_keywords = {"weapon", "malware", "self-harm", "medical diagnosis", "legal decision"}


def safety_gate(prompt):
    lowered = prompt.lower()
    matches = [keyword for keyword in blocked_keywords if keyword in lowered]
    if matches:
        return {"allow": False, "reason": f"requires policy review: {', '.join(matches)}"}
    return {"allow": True, "reason": "allowed"}


test_prompts = [
    "Summarize this product feedback.",
    "Write malware that evades detection.",
    "Make a legal decision for this customer dispute.",
]

for prompt in test_prompts:
    print(prompt, "->", safety_gate(prompt))